<a href="https://colab.research.google.com/github/Kristina-Analyst/Portfolio-Projects/blob/main/Portfolio_Project_2_A_B_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

from google.colab import drive
drive.mount("/content/drive")
%cd /content/drive/MyDrive/MateHometask
df = pd.read_csv("AB-test.csv")
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/MateHometask


,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-01,Lithuania,mobile,Europe,Organic Search,2,2,new account,1
1,2020-11-01,El Salvador,desktop,Americas,Social Search,2,1,new account,1
2,2020-11-01,Slovakia,mobile,Europe,Paid Search,2,2,new account,1
3,2020-11-01,Lithuania,desktop,Europe,Paid Search,2,2,new account,1
4,2020-11-02,North Macedonia,desktop,Europe,Direct,2,1,new account,1


## Розрахунок метрик

In [ ]:
metrics = ['add_payment_info', 'add_shipping_info', 'begin_checkout', 'new account']
df_result = pd.DataFrame()

for number_of_test in (1, 2, 3, 4):
    df1 = df[df['test'] == number_of_test]
    pivot_table = pd.pivot_table(df1, values='value', index='event_name', columns='test_group', aggfunc="sum")

    for n in metrics:
        if n in pivot_table.index and "session" in pivot_table.index:
            if 1 in pivot_table.columns and 2 in pivot_table.columns:
                try:
                    num_conv_A = pivot_table.loc[n, 1]
                    denom_conv_A = pivot_table.loc["session", 1]
                    conv_rate_A = num_conv_A / denom_conv_A

                    num_conv_B = pivot_table.loc[n, 2]
                    denom_conv_B = pivot_table.loc["session", 2]
                    conv_rate_B = num_conv_B / denom_conv_B

                    metric_change = ((conv_rate_B / conv_rate_A) - 1) * 100
                    z_stat, p_value = sm.stats.proportions_ztest([num_conv_A, num_conv_B],
                                                                  [denom_conv_A, denom_conv_B])

                    df_test = pd.DataFrame({
                        'test_number': [number_of_test],
                        'metric': [f"{n}/session"],
                        'numerator_event': [n],
                        'denominator_event': ['session'],
                        'numerator_converse_A': [num_conv_A],
                        'denominator_converse_A': [denom_conv_A],
                        'conversion_rate_A': [conv_rate_A],
                        'numerator_converse_B': [num_conv_B],
                        'denominator_converse_B': [denom_conv_B],
                        'conversion_rate_B': [conv_rate_B],
                        'metric_change': [metric_change],
                        'z_stat': [z_stat],
                        'p_value': [p_value],
                        'significant': [p_value < 0.05]
                    })

                    df_result = pd.concat([df_result, df_test], ignore_index=True)
                except ZeroDivisionError:
                    continue
df_result

,test_number,metric,numerator_event,denominator_event,numerator_converse_A,denominator_converse_A,conversion_rate_A,numerator_converse_B,denominator_converse_B,conversion_rate_B,metric_change,z_stat,p_value,significant
0,1,add_payment_info/session,add_payment_info,session,1988,45362,0.043825,2229,45193,0.049322,12.542021,-3.924884,0.000087,True
1,1,add_shipping_info/session,add_shipping_info,session,3034,45362,0.066884,3221,45193,0.071272,6.560481,-2.603571,0.009226,True
2,1,begin_checkout/session,begin_checkout,session,3784,45362,0.083418,4021,45193,0.088974,6.660587,-2.978783,0.002894,True
3,1,new account/session,new account,session,3823,45362,0.084278,3681,45193,0.081451,-3.354299,1.542883,0.122859,False
4,2,add_payment_info/session,add_payment_info,session,2344,50637,0.046290,2409,50244,0.047946,3.576911,-1.240994,0.214608,False
5,2,add_shipping_info/session,add_shipping_info,session,3480,50637,0.068724,3510,50244,0.069859,1.650995,-0.709557,0.477979,False
6,2,begin_checkout/session,begin_checkout,session,4262,50637,0.084168,4313,50244,0.085841,1.988164,-0.952898,0.340642,False
7,2,new account/session,new account,session,4165,50637,0.082252,4184,50244,0.083274,1.241934,-0.588793,0.556000,False
8,3,add_payment_info/session,add_payment_info,session,3623,70047,0.051722,3697,70439,0.052485,1.474630,-0.643172,0.520112,False
9,3,add_shipping_info/session,add_shipping_info,session,5298,70047,0.075635,5188,70439,0.073652,-2.621211,1.413727,0.157442,False


In [ ]:
df_result.to_excel('result.xlsx', index=False)


In [ ]:
df_result.to_csv('result.csv', index=False)

# Посилання на результат у форматі xlsx:
https://docs.google.com/spreadsheets/d/1Kar3bhyCtQF1a5NgMEFeW3IMSzd1-p4n/edit?usp=sharing&ouid=108913799919106111055&rtpof=true&sd=true

# Посилання на Dashboard:
https://public.tableau.com/app/profile/.38337362/viz/ABtest_17418010700020/ABtest?publish=yes